# 🧠 Notebook B4 — Path B: Train PolyWhisper MoLE Router

**Path B Concrete Specs — Stage 4: Token-Level MoLE Router (The Critical Bottleneck)**

In PolyWhisper's Mixture-of-LoRA-Experts architecture, the router is a compact MLP (~33K parameters) that inspects Whisper's decoder hidden states at every decoding step and predicts per-token which expert to invoke:
- **Expert 0**: Bengali Expert (trained in **Notebook B2**)
- **Expert 1**: English Expert (trained in **Notebook B3**)

> 💡 **Parallel Workflow**: The router is trained on the base Whisper decoder hidden states using the token-tagged code-switched data from Notebook B1. You can train this router while or after the experts finish!

### Benchmark Context:
- **PolyWhisper Router Ceiling**: PolyWhisper's own router achieved **89.1%** per-token routing accuracy. Expect around 85%–89% on Bengali-English code-switched data (roughly 1 in 9 tokens will be misrouted; this is the expected behavior of the architecture).
- **Router Architecture**: Decoder hidden states ($d_{\text{model}} = 1024$) $\to$ Linear(1024, 32) $\to$ GELU $\to$ Linear(32, 2) = **32,866 parameters** (~33K parameters).
- **Frozen Backbone**: The base Whisper model is completely frozen during router training — only the 33K router parameters receive gradients, making training super fast and lightweight (~15 minutes on Colab T4)!

---
⏭️ **Next step:** Once the router and both experts are trained, proceed to **Notebook B5** for the three-way evaluation.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Mount Drive + Prerequisite Readiness Check
# Main Google Drive Folder: https://drive.google.com/drive/folders/1zqBtJvJ8ZwZNNpLCiS2F-x-yBpMxYXfE?usp=sharing
# data_path_b Folder: https://drive.google.com/drive/folders/1hbAfVrgYEJpASZGxQmNX0zaEok3ry19D?usp=sharing
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
DRIVE_ROOT      = '/content/drive/MyDrive/hoichoi'
DATA_DIR        = DRIVE_ROOT + '/data_path_b'
BN_ADAPTER      = DRIVE_ROOT + '/adapters_path_b/bengali_expert/best'
EN_ADAPTER      = DRIVE_ROOT + '/adapters_path_b/english_expert/best'
ROUTER_OUT      = DRIVE_ROOT + '/router_path_b'
DATA_B_DIR_ID   = '1hbAfVrgYEJpASZGxQmNX0zaEok3ry19D'

os.makedirs(DATA_DIR, exist_ok=True)

# Direct check for tagged dataset in data_path_b (download if missing)
if not os.path.exists(DATA_DIR + '/train_router_tagged.json'):
    print(f'📥 Fetching data_path_b into {DATA_DIR} from Drive folder link...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
    import gdown
    try:
        gdown.download_folder(id=DATA_B_DIR_ID, output=DATA_DIR, quiet=False, use_cookies=False)
    except Exception as e:
        print('Download notice:', e)

print('Checking prerequisite readiness for Stage 4 Router ...')
bn_ready   = os.path.exists(BN_ADAPTER + '/adapter_config.json')
en_ready   = os.path.exists(EN_ADAPTER + '/adapter_config.json')
data_ready = os.path.exists(DATA_DIR + '/train_router_tagged.json')

print(f'  Bengali Expert (Notebook B2) : {"✅ Ready" if bn_ready else "⏳ Training / Pending"}')
print(f'  English Expert (Notebook B3) : {"✅ Ready" if en_ready else "⏳ Training / Pending"}')
print(f'  Tagged Data    (Notebook B1) : {"✅ Ready" if data_ready else "❌ MISSING"}')

if not data_ready:
    raise RuntimeError(
        f'❌ Gate check failed: Token-tagged data (train_router_tagged.json) not found in {DATA_DIR}! Run Notebook B1 first.'
    )

if not (bn_ready and en_ready):
    print('\n💡 Notice: One or both LoRA expert adapters have not finished training yet.')
    print('   You can STILL train this Router now because it trains on base Whisper hidden states!')
    print('   (Both adapters are only required later for Notebook B5 inference.)')

os.makedirs(ROUTER_OUT, exist_ok=True)
print('\n✅ Prerequisite checks passed! Router output directory:', ROUTER_OUT)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Dependencies & GPU Verification
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, sys
print('Installing transformers, peft, accelerate, librosa, soundfile ...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40.0', 'peft>=0.10.0,<0.14.0', 'accelerate>=0.29.0',
    'librosa', 'soundfile'])
print('✅ Packages installed.')

try:
    import peft.import_utils as _piu
    _piu.is_torchao_available = lambda: False
except Exception:
    pass

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU Detected: {gpu_name} ({vram_gb:.1f} GB VRAM)')
else:
    print('⚠️ No GPU found! Please navigate to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Router Dataset with Token-Level Language Supervision
# Loads exact audio segment slice via start_sec & end_sec
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, json, librosa
import numpy as np
import torch
from torch.utils.data import Dataset

class RouterDataset(Dataset):
    def __init__(self, json_path, feature_extractor, tokenizer, sr=16000):
        with open(json_path, encoding='utf-8') as f:
            self.records = json.load(f)
        self.fe  = feature_extractor
        self.tok = tokenizer
        self.sr  = sr
        self.max_audio_len = 30 * sr
        
        # Prefix prompt tokens for Bengali transcription
        self.prefix_ids = self.tok.prefix_tokens
        self.eos_token_id = self.tok.eos_token_id

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records[idx]
        s_sec = float(row.get('start_sec', 0.0))
        e_sec = float(row.get('end_sec', 0.0))
        dur   = (e_sec - s_sec) if e_sec > s_sec else None
        
        apath = str(row.get('audio_path', ''))
        if not os.path.exists(apath):
            fname = os.path.basename(apath)
            for cdir in [
                '/content/drive/MyDrive/hoichoi/mucs_raw/train',
                '/content/drive/MyDrive/hoichoi/mucs_raw',
                '/content/drive/MyDrive/mucs_raw/train',
                '/content/drive/MyDrive/mucs_raw',
                '/content/mucs_raw/train',
                '/content/mucs_raw',
                '/content'
            ]:
                candidate = os.path.join(cdir, fname)
                if os.path.exists(candidate):
                    apath = candidate
                    break
        
        try:
            audio, _ = librosa.load(apath, sr=self.sr, mono=True,
                                    offset=s_sec, duration=min(30.0, dur) if dur else 30.0)
            if len(audio) == 0:
                audio = np.zeros(self.sr, dtype=np.float32)
        except Exception:
            audio = np.zeros(self.sr, dtype=np.float32)
        
        audio = audio[:self.max_audio_len]
        feats = self.fe(audio, sampling_rate=self.sr, return_tensors='pt').input_features.squeeze(0)
        
        # Teacher forcing sequence: [prefix_ids] + [content_tokens] + [eos_token_id]
        content_tokens = row['token_ids']
        content_langs  = row['token_langs']
        
        dec_input_ids = list(self.prefix_ids) + content_tokens + [self.eos_token_id]
        # Language supervision targets (-100 for prefix & EOS so loss is ignored)
        target_langs  = ([-100] * len(self.prefix_ids)) + content_langs + [-100]
        
        # Whisper max decoder position embedding length is 448
        dec_input_ids = dec_input_ids[:448]
        target_langs  = target_langs[:448]
        
        return {
            'input_features': feats,
            'decoder_input_ids': torch.tensor(dec_input_ids, dtype=torch.long),
            'target_langs': torch.tensor(target_langs, dtype=torch.long)
        }

def collate_router(batch):
    feats = torch.stack([b['input_features'] for b in batch])
    dec_ids = [b['decoder_input_ids'] for b in batch]
    targets = [b['target_langs'] for b in batch]
    
    maxlen = max(t.size(0) for t in dec_ids)
    
    padded_dec = torch.full((len(dec_ids), maxlen), 50257, dtype=torch.long)  # 50257 = pad/eos
    padded_tgt = torch.full((len(targets), maxlen), -100, dtype=torch.long)   # -100 = ignore
    
    for i in range(len(batch)):
        l = dec_ids[i].size(0)
        padded_dec[i, :l] = dec_ids[i]
        padded_tgt[i, :l] = targets[i]
        
    return {
        'input_features': feats,
        'decoder_input_ids': padded_dec,
        'target_langs': padded_tgt
    }

print('✅ Router Dataset and collator configured')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — Router MLP Architecture (~33K Parameters)
# Over Decoder Hidden States (d_model = 1024)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
import torch.nn as nn
from transformers import WhisperForConditionalGeneration, WhisperProcessor

class WhisperMoERouter(nn.Module):
    """
    PolyWhisper Mixture-of-LoRA-Experts Router:
    Takes decoder hidden state at each token (dim=1024) and predicts
    routing logits for [Bengali Expert (0), English Expert (1)].
    Total parameters: ~33K.
    """
    def __init__(self, d_model=1024, hidden_dim=32, num_experts=2, dropout=0.1):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_experts)
        )

    def forward(self, decoder_hidden_states):
        # decoder_hidden_states: [batch_size, seq_len, d_model]
        return self.mlp(decoder_hidden_states)
    
device = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE_MODEL_ID = 'SayedShaun/bengali-whisper-medium'

print('Loading processor and base Whisper model ...')
processor = WhisperProcessor.from_pretrained(BASE_MODEL_ID, language='Bengali', task='transcribe')
base_whisper = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.float16 if device == 'cuda' else torch.float32
).to(device)

# Freeze the base model completely (Zero gradients for backbone)
for param in base_whisper.parameters():
    param.requires_grad = False
base_whisper.eval()

# Instantiate the 33K parameter router
router = WhisperMoERouter(d_model=1024, hidden_dim=32, num_experts=2, dropout=0.1).to(device)
n_params = sum(p.numel() for p in router.parameters() if p.requires_grad)
print(f'✅ Router Architecture initialized: {n_params:,} trainable parameters (PolyWhisper ~33K spec)')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — Train the MoLE Router
# Tracks Per-Token Accuracy against PolyWhisper's 89.1% Ceiling
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, json
from torch.utils.data import DataLoader

# Hyperparameters
ROUTER_CFG = dict(
    batch_size    = 8,
    grad_accum    = 2,
    lr            = 1e-3,
    epochs        = 10,
    val_every     = 100,
    ceiling_acc   = 0.891,  # PolyWhisper reference ceiling
)

train_json = DATA_DIR + '/train_router_tagged.json'
val_json   = DATA_DIR + '/val_router_tagged.json'

train_ds = RouterDataset(train_json, processor.feature_extractor, processor.tokenizer)
val_ds   = RouterDataset(val_json,   processor.feature_extractor, processor.tokenizer)

# Set num_workers=0 to prevent multiprocessing hanging on Colab
train_dl = DataLoader(train_ds, batch_size=ROUTER_CFG['batch_size'], shuffle=True,  collate_fn=collate_router, num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=ROUTER_CFG['batch_size'], shuffle=False, collate_fn=collate_router, num_workers=0)

criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.AdamW(router.parameters(), lr=ROUTER_CFG['lr'], weight_decay=0.01)
scaler    = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))

def eval_router():
    router.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch in val_dl:
            feats = batch['input_features'].to(device)
            dec_ids = batch['decoder_input_ids'].to(device)
            targets = batch['target_langs'].to(device)
            
            with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                dec_out = base_whisper(input_features=feats, decoder_input_ids=dec_ids, output_hidden_states=True)
                hidden  = dec_out.decoder_hidden_states[-1]
                logits  = router(hidden.float())
                loss    = criterion(logits.view(-1, 2), targets.view(-1))
            
            val_loss += loss.item() * feats.size(0)
            preds = logits.argmax(dim=-1)
            mask  = (targets != -100)
            correct += ((preds == targets) & mask).sum().item()
            total   += mask.sum().item()
    
    router.train()
    avg_loss = val_loss / max(len(val_ds), 1)
    acc      = correct / max(total, 1)
    return avg_loss, acc

print('=' * 65)
print(f'Starting Router Training — Reference Ceiling: {ROUTER_CFG["ceiling_acc"]*100:.1f}% accuracy')
print('=' * 65)

best_acc   = 0.0
step_count = 0
accum_step = 0
router.train()

for epoch in range(1, ROUTER_CFG['epochs'] + 1):
    for batch in train_dl:
        feats   = batch['input_features'].to(device, non_blocking=True)
        dec_ids = batch['decoder_input_ids'].to(device, non_blocking=True)
        targets = batch['target_langs'].to(device, non_blocking=True)
        
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                dec_out = base_whisper(input_features=feats, decoder_input_ids=dec_ids, output_hidden_states=True)
                hidden  = dec_out.decoder_hidden_states[-1]
        
        with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
            logits = router(hidden.float())
            loss   = criterion(logits.view(-1, 2), targets.view(-1)) / ROUTER_CFG['grad_accum']
            
        scaler.scale(loss).backward()
        accum_step += 1
        
        if accum_step % ROUTER_CFG['grad_accum'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(router.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            step_count += 1
            accum_step = 0
            
            if step_count % 20 == 0:
                print(f'  Epoch {epoch:02d} | Step {step_count:>4} | Loss: {loss.item()*ROUTER_CFG["grad_accum"]:.4f}')
                
            if step_count % ROUTER_CFG['val_every'] == 0:
                v_loss, v_acc = eval_router()
                print(f'\n  ✦ Validation [Step {step_count}]: Loss = {v_loss:.4f} | Per-Token Accuracy = {v_acc*100:.2f}% (Ceiling: 89.1%)')
                
                if v_acc > best_acc:
                    best_acc = v_acc
                    best_dir = ROUTER_OUT + '/best'
                    os.makedirs(best_dir, exist_ok=True)
                    torch.save(router.state_dict(), best_dir + '/router.pt')
                    with open(best_dir + '/router_config.json', 'w') as f:
                        json.dump({
                            'd_model': 1024,
                            'hidden_dim': 32,
                            'num_experts': 2,
                            'best_val_accuracy': best_acc,
                            'bengali_adapter': BN_ADAPTER,
                            'english_adapter': EN_ADAPTER,
                        }, f, indent=2)
                    print(f'    ✅ New Best Router Saved! Accuracy: {best_acc*100:.2f}%\n')

# Final evaluation
v_loss, v_acc = eval_router()
if v_acc > best_acc or not os.path.exists(ROUTER_OUT + '/best/router.pt'):
    best_acc = v_acc
    best_dir = ROUTER_OUT + '/best'
    os.makedirs(best_dir, exist_ok=True)
    torch.save(router.state_dict(), best_dir + '/router.pt')
    with open(best_dir + '/router_config.json', 'w') as f:
        json.dump({
            'd_model': 1024,
            'hidden_dim': 32,
            'num_experts': 2,
            'best_val_accuracy': best_acc,
            'bengali_adapter': BN_ADAPTER,
            'english_adapter': EN_ADAPTER,
        }, f, indent=2)

print('=' * 65)
print(f'✅ Router Training Finished! Final Best Accuracy: {best_acc*100:.2f}%')
print(f'   PolyWhisper Reference Ceiling : 89.1%')
print(f'   Router Weights Saved → {ROUTER_OUT}/best/router.pt')
print('=' * 65)
print('👉 Next: Open Notebook B5 (Evaluate & Compare) to run the 3-way benchmark!')